# Dataset comparison: real vs. generated

Computes statistics over the full real BraTS-2023 dataset (1251 T1N volumes and segmentation masks) and compares them to the generated outputs.

Two figures:
- `intensity_histogram_real_vs_generated.pdf` — voxel intensity distribution per model vs. real
- `tumor_size_distribution.pdf` — tumor voxel-count distribution (total + per subregion) from real seg masks vs. pipeline-generated masks

The real-dataset stats are expensive to compute (one full pass over 1251 volumes), so they are cached to `output/real_dataset_stats.json`. Re-runs use the cache automatically.

## Configuration

In [ ]:
from pathlib import Path

# Reale BraTS-Daten
BRATS_REAL_ROOT = Path(r"C:\Users\Sven\Desktop\Data\Processed\brats\32")
BRATS_SEG_ROOT  = Path(r"C:\Users\Sven\Desktop\Data\Processed\brats\32_seg")
# Linux: Path("/home/sven/Desktop/Data/Processed/brats/32") usw.

# Generierte Samples
SAMPLES_UNCOND_ROOT   = Path(r"C:\Users\Sven\Desktop\T3_3200\3DMedDiffusion\Code\unconditional\samples")
SAMPLES_PIPELINE_ROOT = Path(r"C:\Users\Sven\Desktop\T3_3200\3DMedDiffusion\Code\conditional\pipeline")

# Output
OUTPUT_DIR = Path.cwd() / "output"
OUTPUT_DIR.mkdir(exist_ok=True)

# FID-bester Sampler je Modell (für die Intensitäts-Verteilungen)
FID_BEST_SAMPLER = {
    "baseline":          "DDPM",
    "data_augmentation": "UniPC",
    "linear_schedule":   "DDPM",
    "no_attention":      "DDPM",
    "noise_prediction":  "DDPM",
}
PIPELINE_SAMPLER = "UniPC"

print(f"output: {OUTPUT_DIR}")

## Style and imports

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt
import matplotlib as mpl
import nibabel as nib
from tqdm.auto import tqdm

mpl.rcParams.update({
    "font.family": "serif",
    "font.size": 9,
    "axes.labelsize": 9,
    "axes.titlesize": 10,
    "legend.fontsize": 8,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "lines.linewidth": 1.2,
    "savefig.dpi": 300,
    "savefig.bbox": "tight",
})

MODEL_COLORS = {
    "baseline":          "#1f77b4",
    "data_augmentation": "#ff7f0e",
    "linear_schedule":   "#d62728",
    "no_attention":      "#2ca02c",
    "noise_prediction":  "#9467bd",
}

MODEL_LABELS = {
    "baseline":          r"Baseline $32^3$",
    "data_augmentation": "Data Augmentation",
    "linear_schedule":   "Linearer Schedule",
    "no_attention":      "Ohne Self-Attention",
    "noise_prediction":  "Noise-Prediction",
}

def load_volume(p):
    return nib.load(str(p)).get_fdata().astype(np.float32)

print("ready")

## Real dataset statistics (cached)
Iteriert einmalig über alle realen Volumina und Masken und cached das Ergebnis. Re-Runs lesen den Cache.

In [ ]:
REAL_STATS_CACHE = OUTPUT_DIR / "real_dataset_stats.json"
INTENSITY_BIN_EDGES = np.linspace(-1.0, 1.0, 51)  # gleich wie in den generierten stats.json

def compute_real_dataset_stats():
    vol_files = sorted(BRATS_REAL_ROOT.glob("*.nii.gz"))
    seg_files = sorted(BRATS_SEG_ROOT.glob("*.nii.gz"))
    print(f"reale Daten: {len(vol_files)} T1N, {len(seg_files)} Masken")

    # Intensitätshistogramm akkumulieren
    hist_counts = np.zeros(len(INTENSITY_BIN_EDGES) - 1, dtype=np.int64)
    voxel_sum, voxel_sq_sum, voxel_n = 0.0, 0.0, 0
    for vp in tqdm(vol_files, desc="T1N"):
        vol = load_volume(vp).ravel()
        h, _ = np.histogram(vol, bins=INTENSITY_BIN_EDGES)
        hist_counts += h
        voxel_sum    += float(vol.sum())
        voxel_sq_sum += float((vol.astype(np.float64) ** 2).sum())
        voxel_n      += vol.size

    global_mean = voxel_sum / voxel_n
    global_std  = float(np.sqrt(voxel_sq_sum / voxel_n - global_mean ** 2))

    # Tumor-Statistik
    total_voxels = []
    class_voxels = {1: [], 2: [], 3: []}    # nekrotischer Tumorkern, peritumorales Ödem, anreichernder Tumor
    bbox_dims_list = []
    n_empty = 0
    for sp in tqdm(seg_files, desc="Seg"):
        seg = load_volume(sp).astype(np.int8)
        total = int((seg > 0).sum())
        total_voxels.append(total)
        if total == 0:
            n_empty += 1
            continue
        for lab in (1, 2, 3):
            class_voxels[lab].append(int((seg == lab).sum()))
        nz = np.where(seg > 0)
        bbox_dims_list.append([int(nz[0].max() - nz[0].min() + 1),
                               int(nz[1].max() - nz[1].min() + 1),
                               int(nz[2].max() - nz[2].min() + 1)])

    return {
        "n_volumes": len(vol_files),
        "n_segmentations": len(seg_files),
        "n_empty_segmentations": n_empty,
        "intensity": {
            "bin_edges": INTENSITY_BIN_EDGES.tolist(),
            "counts":    hist_counts.tolist(),
            "global_mean": global_mean,
            "global_std":  global_std,
        },
        "tumor": {
            "total_voxels": total_voxels,
            "class_voxels": {str(k): v for k, v in class_voxels.items()},
            "bbox_dims":    bbox_dims_list,
            "total_mean":   float(np.mean(total_voxels)),
            "total_median": float(np.median(total_voxels)),
            "total_min":    int(np.min(total_voxels)),
            "total_max":    int(np.max(total_voxels)),
            "class_mean":   {str(k): float(np.mean(v)) for k, v in class_voxels.items() if v},
        },
    }

if REAL_STATS_CACHE.exists():
    with REAL_STATS_CACHE.open() as f:
        real_stats = json.load(f)
    print(f"Cache geladen: {REAL_STATS_CACHE.name}")
else:
    real_stats = compute_real_dataset_stats()
    with REAL_STATS_CACHE.open("w") as f:
        json.dump(real_stats, f, indent=2)
    print(f"berechnet und gespeichert: {REAL_STATS_CACHE.name}")

print(f"  N Volumina: {real_stats['n_volumes']}")
print(f"  Intensität: mean={real_stats['intensity']['global_mean']:.4f}  std={real_stats['intensity']['global_std']:.4f}")
print(f"  Tumor (gesamt): mean={real_stats['tumor']['total_mean']:.1f} Voxel  median={real_stats['tumor']['total_median']:.1f}")
print(f"  Tumor pro Subregion: {real_stats['tumor']['class_mean']}")
print(f"  Leere Segmentierungen: {real_stats['n_empty_segmentations']}")

## Intensity histogram: real vs generated
Überlagert die Intensitätsverteilung der realen BraTS-T1N-Volumina mit den generierten Verteilungen pro Modell.

In [ ]:
def load_generated_intensity_hist(model, sampler, size=32):
    p = SAMPLES_UNCOND_ROOT / str(size) / sampler / f"samples_{model}" / "stats.json"
    if not p.exists():
        return None
    with p.open() as f:
        st = json.load(f)
    vd = st.get("aggregates", {}).get("voxel_distribution")
    if vd is None:
        return None
    return np.array(vd["bin_edges"]), np.array(vd["counts"])

def to_density(edges, counts):
    widths = np.diff(edges)
    total = counts.sum()
    return (counts / total) / widths if total > 0 else counts

fig, ax = plt.subplots(1, 1, figsize=(7.5, 4.0))

# Reale Verteilung als gefüllter Hintergrund
edges_r  = np.array(real_stats["intensity"]["bin_edges"])
counts_r = np.array(real_stats["intensity"]["counts"])
density_r = to_density(edges_r, counts_r)
centers_r = 0.5 * (edges_r[:-1] + edges_r[1:])
ax.fill_between(centers_r, density_r, color="gray", alpha=0.25, label="BraTS (real)")
ax.plot(centers_r, density_r, color="black", linewidth=1.2)

# Generierte Verteilungen pro Modell
for m, sampler in FID_BEST_SAMPLER.items():
    res = load_generated_intensity_hist(m, sampler, 32)
    if res is None:
        print(f"WARN: keine Verteilung für {m}/{sampler}")
        continue
    edges_g, counts_g = res
    density_g = to_density(edges_g, counts_g)
    centers_g = 0.5 * (edges_g[:-1] + edges_g[1:])
    ax.plot(centers_g, density_g, color=MODEL_COLORS[m],
            label=f"{MODEL_LABELS[m]} ({sampler})", linewidth=1.1)

ax.set_xlabel("Voxel-Intensität")
ax.set_ylabel("Dichte")
ax.set_xlim(-1.0, 1.0)
ax.set_yscale("log")
ax.grid(True, which="both", alpha=0.3, linewidth=0.5)
ax.legend(loc="upper right", fontsize=7.5, frameon=True)
fig.tight_layout()
fig.savefig(OUTPUT_DIR / "intensity_histogram_real_vs_generated.pdf")
plt.show()
print("→ intensity_histogram_real_vs_generated.pdf")

## Tumor size distribution: real vs generated
Vergleicht die Verteilung der Tumor-Voxelanzahl zwischen realen BraTS-Segmentierungen und den von der Pipeline erzeugten Masken — links Gesamt, rechts pro Subregion.

In [ ]:
def load_pipeline_tumor_sizes(sampler="UniPC", size=32):
    p = SAMPLES_PIPELINE_ROOT / sampler / str(size) / "stats.json"
    if not p.exists():
        return None
    with p.open() as f:
        st = json.load(f)
    ps = st.get("per_sample", [])
    sizes_total = np.array([s.get("tumor_voxel_count", 0) for s in ps])
    classes = {1: [], 2: [], 3: []}
    for s in ps:
        per_cls = s.get("tumor_voxel_count_per_class", [0, 0, 0])
        for i, v in enumerate(per_cls, start=1):
            classes[i].append(v)
    return sizes_total, {k: np.array(v) for k, v in classes.items()}

real_total   = np.array(real_stats["tumor"]["total_voxels"])
real_classes = {int(k): np.array(v) for k, v in real_stats["tumor"]["class_voxels"].items()}

pipe = load_pipeline_tumor_sizes(PIPELINE_SAMPLER, 32)
if pipe is None:
    print("WARN: keine Pipeline-Stats")
else:
    gen_total, gen_classes = pipe
    fig, axes = plt.subplots(1, 2, figsize=(11, 4.0))

    # Links: Gesamttumor-Größe
    ax = axes[0]
    bins = np.linspace(0, max(real_total.max(), gen_total.max()), 40)
    ax.hist(real_total, bins=bins, alpha=0.5, color="gray",
            label=f"BraTS (real), N={len(real_total)}", density=True)
    ax.hist(gen_total,  bins=bins, alpha=0.5, color=MODEL_COLORS["baseline"],
            label=f"Pipeline {PIPELINE_SAMPLER}, N={len(gen_total)}", density=True)
    ax.set_xlabel("Tumor-Voxelanzahl (gesamt)")
    ax.set_ylabel("Dichte")
    ax.legend(frameon=True)
    ax.grid(True, alpha=0.3, linewidth=0.5)

    # Rechts: Box-Plot pro Subregion
    ax = axes[1]
    labels_cls = ["Nekrotischer\nTumorkern\n(Label 1)", "Peritumorales\nÖdem\n(Label 2)", "Anreichernder\nTumor\n(Label 3)"]
    pos_real = np.array([1, 4, 7])
    pos_gen  = np.array([2, 5, 8])
    data_real = [real_classes[k] for k in (1, 2, 3)]
    data_gen  = [gen_classes[k]  for k in (1, 2, 3)]

    bp1 = ax.boxplot(data_real, positions=pos_real, widths=0.7, patch_artist=True,
                     medianprops={"color": "black"}, showfliers=False)
    bp2 = ax.boxplot(data_gen, positions=pos_gen, widths=0.7, patch_artist=True,
                     medianprops={"color": "black"}, showfliers=False)
    for patch in bp1["boxes"]:
        patch.set_facecolor("gray"); patch.set_alpha(0.5)
    for patch in bp2["boxes"]:
        patch.set_facecolor(MODEL_COLORS["baseline"]); patch.set_alpha(0.5)
    ax.set_xticks(0.5 * (pos_real + pos_gen))
    ax.set_xticklabels(labels_cls)
    ax.set_ylabel("Voxelanzahl pro Subregion")
    ax.grid(True, axis="y", alpha=0.3, linewidth=0.5)
    from matplotlib.patches import Patch
    ax.legend(handles=[Patch(facecolor="gray", alpha=0.5, label="BraTS (real)"),
                       Patch(facecolor=MODEL_COLORS["baseline"], alpha=0.5, label=f"Pipeline {PIPELINE_SAMPLER}")],
              frameon=True)

    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "tumor_size_distribution.pdf")
    plt.show()
    print("→ tumor_size_distribution.pdf")

    print(f"\nReal:      mean={real_total.mean():.1f}  median={np.median(real_total):.1f}  max={real_total.max()}")
    print(f"Generiert: mean={gen_total.mean():.1f}  median={np.median(gen_total):.1f}  max={gen_total.max()}")

## Done

In [ ]:
print("outputs unter:", OUTPUT_DIR)